# DockerAgent — Fine-Tuning Phi-3-mini on Google Colab

**Model:** `microsoft/Phi-3-mini-4k-instruct` (3.8B params)  
**Method:** QLoRA (4-bit quantization + LoRA adapters)  
**GPU required:** T4 (16GB VRAM) — free on Colab  
**Estimated time:** 1–2 hours  

## Before running:
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Run cells top to bottom
3. When asked to upload files, upload `train.jsonl` and `val.jsonl` from `agent/dataset/`

## Cell 1 — Install libraries
These are not pre-installed on Colab. `bitsandbytes` enables 4-bit quantization. `peft` provides LoRA. `trl` provides the `SFTTrainer` for instruction fine-tuning.

In [ ]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate scipy

## Cell 2 — Mount Google Drive
We save model weights to Drive so they persist after the Colab session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/docker_agent_model'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Model will be saved to: {OUTPUT_DIR}')

## Cell 3 — Upload training data
Upload your `train.jsonl` and `val.jsonl` files from `agent/dataset/` on your laptop.

In [ ]:
from google.colab import files

print('Upload train.jsonl and val.jsonl')
uploaded = files.upload()

for fname in uploaded:
    print(f'Uploaded: {fname} ({len(uploaded[fname])} bytes)')

# Verify both files are present
import os
assert os.path.exists('train.jsonl'), 'train.jsonl not found!'
assert os.path.exists('val.jsonl'),   'val.jsonl not found!'
print('Both files ready.')

## Cell 4 — Load and inspect dataset
We load the JSONL files and check the format. Each example must have a `conversations` key with `system`, `user`, `assistant` turns.

In [ ]:
from datasets import load_dataset
import json

dataset = load_dataset('json', data_files={
    'train': 'train.jsonl',
    'validation': 'val.jsonl'
})

print(f'Train examples:      {len(dataset["train"])}')
print(f'Validation examples: {len(dataset["validation"])}')
print()
print('First example preview:')
ex = dataset['train'][0]
for turn in ex['conversations'][:4]:
    print(f'  [{turn["role"]}]: {turn["content"][:80]}...' if len(turn['content']) > 80 else f'  [{turn["role"]}]: {turn["content"]}')

## Cell 5 — Load base model in 4-bit (QLoRA)
We load `Phi-3-mini-4k-instruct` quantized to 4-bit using `bitsandbytes`. This reduces VRAM from ~8GB to ~2.5GB, leaving room for training.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoConfig

MODEL_ID = 'microsoft/Phi-3-mini-4k-instruct'

# Patch: if rope_scaling doesn't have 'type', check rope_type value.
# For the 4k model, rope_type='default' means no scaling → set rope_scaling=None.
config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
if hasattr(config, 'rope_scaling') and isinstance(config.rope_scaling, dict):
    if 'type' not in config.rope_scaling:
        rope_type = config.rope_scaling.get('rope_type', 'default')
        if rope_type == 'longrope':
            config.rope_scaling['type'] = 'longrope'
        else:
            config.rope_scaling = None  # standard rotary, no scaling needed

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading {MODEL_ID} in 4-bit...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    config=config,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Model loaded.')
print(f'GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## Cell 6 — Configure LoRA adapters
`r=16` controls adapter size (bigger = more capacity, more VRAM). `alpha=32` is the scaling factor. We target `q_proj` and `v_proj` — the attention projection layers that matter most for conversational behavior.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['qkv_proj', 'o_proj'],  # Phi-3 uses combined qkv_proj, not separate q/v
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected output: ~0.5-1% trainable (very efficient)

## Cell 7 — Format dataset into Phi-3 chat template
Phi-3 uses a specific prompt format: `<|system|>...<|end|><|user|>...<|end|><|assistant|>...`  
We convert each `conversations` list into this format before tokenizing.

In [ ]:
def format_phi3_conversation(example):
    """
    Convert our conversations format to Phi-3 prompt string.
    Phi-3 format:
      <|system|>\n{system}<|end|>\n<|user|>\n{user}<|end|>\n<|assistant|>\n{assistant}<|end|>
    """
    turns = example['conversations']
    prompt = ''

    system_content = ''
    for turn in turns:
        if turn['role'] == 'system':
            system_content = turn['content']
            prompt += f"<|system|>\n{system_content}<|end|>\n"
        elif turn['role'] == 'user':
            prompt += f"<|user|>\n{turn['content']}<|end|>\n"
        elif turn['role'] == 'assistant':
            prompt += f"<|assistant|>\n{turn['content']}<|end|>\n"

    return {'text': prompt}


formatted_train = dataset['train'].map(format_phi3_conversation)
formatted_val   = dataset['validation'].map(format_phi3_conversation)

# Preview
print('=== Formatted example (first 500 chars) ===')
print(formatted_train[0]['text'][:500])

## Cell 8 — Training configuration
Key settings explained:
- `num_train_epochs=3` — 3 passes over the dataset (good baseline)
- `per_device_train_batch_size=2` — 2 examples per GPU step (T4 can handle this)
- `gradient_accumulation_steps=4` — effective batch = 2×4 = 8
- `max_seq_length=1024` — cap conversation length to save VRAM
- `save_steps=50` — checkpoint every 50 steps (safety net)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='/content/checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    warmup_steps=20,
    lr_scheduler_type='cosine',
    report_to='none',
    optim='paged_adamw_8bit',
)

def formatting_func(example):
    return example['text']

class PatchedSFTTrainer(SFTTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        return super().compute_loss(model, inputs, return_outputs=return_outputs)

trainer = PatchedSFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=formatted_train,
    eval_dataset=formatted_val,
    formatting_func=formatting_func,
    args=training_args,
)

print('Trainer configured. Ready to train.')
print(f'Total training steps: {trainer.state.max_steps if hasattr(trainer.state, "max_steps") else "(calculated at start)"}')

## Cell 9 — Run training
This is the main training cell. Expected duration: **~1–2 hours** on T4.  
You will see `loss` decreasing — target is below 0.5 by the end.

In [ ]:
print('Starting training...')
trainer.train()
print('Training complete!')

## Cell 10 — Save LoRA adapter to Google Drive
This saves only the adapter weights (~30MB), not the full model. Faster to download.

In [ ]:
ADAPTER_DIR = OUTPUT_DIR + '/lora_adapter'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'LoRA adapter saved to: {ADAPTER_DIR}')

# List saved files
import os
for f in os.listdir(ADAPTER_DIR):
    size = os.path.getsize(os.path.join(ADAPTER_DIR, f))
    print(f'  {f}: {size/1e6:.1f} MB')

## Cell 11 — (Optional) Merge adapter into full model and save
This merges the LoRA adapter back into the base model weights, producing a standalone model.  
**Only run if you have enough Drive space (~3GB).** The merged model can be loaded without PEFT.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoConfig
import torch

print('Loading base model on CPU for merging...')

config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
if hasattr(config, 'rope_scaling') and isinstance(config.rope_scaling, dict):
    if 'type' not in config.rope_scaling:
        rope_type = config.rope_scaling.get('rope_type', 'default')
        if rope_type == 'longrope':
            config.rope_scaling['type'] = 'longrope'
        else:
            config.rope_scaling = None

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    config=config,
    torch_dtype=torch.float16,
    device_map={'': 'cpu'},  # load on CPU to avoid VRAM offload issues
    trust_remote_code=True,
)

print('Merging adapter into base model...')
merged_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR, device_map={'': 'cpu'})
merged_model = merged_model.merge_and_unload()

# Patch: Phi-3 sets _tied_weights_keys as a list but transformers expects a dict
for module in merged_model.modules():
    if hasattr(module, '_tied_weights_keys') and isinstance(module._tied_weights_keys, list):
        module._tied_weights_keys = {k: k for k in module._tied_weights_keys}

MERGED_DIR = OUTPUT_DIR + '/merged_model'
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f'Merged model saved to: {MERGED_DIR}')

## Cell 12 — Quick inference test (sanity check)
Before downloading, verify the fine-tuned model gives sensible responses.

In [ ]:
SYSTEM_PROMPT = (
    "You are DockerAgent, an intelligent assistant that helps manage Docker Compose services "
    "for a monitoring pipeline project. You can add or remove services. When adding, you collect: "
    "service name, image, port, monitoring probe type, volumes (optional), depends_on (optional). "
    "When removing, you ask for double confirmation. You always generate valid docker-compose YAML "
    "with the project network (monitoring_default) and correct labels. Never guess missing information — always ask."
)

test_prompt = f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n<|user|>\nadd a service called my-api<|end|>\n<|assistant|>\n"

inputs = tokenizer(test_prompt, return_tensors='pt').to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        use_cache=False,
    )

response = tokenizer.decode(output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('=== Model Response ===')
print(response)
# Expected: should ask about the image, not hallucinate one

## Cell 13 — Download adapter (optional — use Drive instead)
If you prefer to download directly to your laptop rather than via Drive.

In [ ]:
import shutil
from google.colab import files

ADAPTER_DIR = '/content/drive/MyDrive/docker_agent_model/lora_adapter'

shutil.make_archive('/content/docker_agent_adapter', 'zip', ADAPTER_DIR)
files.download('/content/docker_agent_adapter.zip')
print('Download started.')

---
## After downloading the adapter

Place the adapter folder at `agent/model/lora_adapter/` on your laptop.

Then in `agent/llm_client.py` (Step 5), load it with:
```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base = AutoModelForCausalLM.from_pretrained('microsoft/Phi-3-mini-4k-instruct', ...)
model = PeftModel.from_pretrained(base, 'agent/model/lora_adapter')
```

Or if you saved the merged model, load it directly:
```python
model = AutoModelForCausalLM.from_pretrained('agent/model/merged_model', ...)
```